#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [ ]:
import pandas as pd 
from importlib import reload
import random

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

import Algoritmos.greedy as greedy_module
reload(greedy_module)
from Algoritmos.greedy import solucion_greedy

import Algoritmos.relocate as relocate_module
reload(relocate_module)
from Algoritmos.relocate import relocate

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")

cajas_nuevas = pd.read_csv("4r.cajas_nuevas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_3mm.csv")

In [215]:
grosor = 3

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [216]:
def guardar_cajas_y_productos(grosor=grosor):
    
    cajas = {
        row["caja_tipo_id"]: Caja(
            caja_id=row["caja_tipo_id"],
            dim_interior_ancho=row["caja_interior_ancho"],
            dim_interior_largo=row["caja_interior_largo"],
            dim_interior_alto=row["caja_interior_alto"]
        )
        for _, row in cajas_nuevas.iterrows()
    }

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
        row["codigo_producto"]: Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    }
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

In [217]:
def ordenar_por_cajas_asignables(asignaciones_por_producto):
    '''
    Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
    '''
    productos_conteo = []

    for codigo_producto, cajas_asignables in asignaciones_por_producto.items():
        cantidad_cajas = len(cajas_asignables)  # Número de cajas asignables para este producto    
        productos_conteo.append({
            'codigo_producto': codigo_producto,
            'cantidad_cajas_asignables': cantidad_cajas
        })

    # Ordenar de menor a mayor cantidad de cajas
    productos_ordenados = sorted(
        productos_conteo,
        key=lambda x: x['cantidad_cajas_asignables'],
        reverse=False
    )

    lista_productos_ordenados = [item['codigo_producto'] for item in productos_ordenados]
    return lista_productos_ordenados

In [218]:
cajas, productos, asignaciones_por_producto = guardar_cajas_y_productos()
lista_productos_ordenados = ordenar_por_cajas_asignables(asignaciones_por_producto)

solucion_greedy = solucion_greedy(cajas, productos, lista_productos_ordenados, asignaciones_por_producto,
                                  criterio_greedy="maximizar_utilizacion_pallet",
                                  titulo_solucion="Greedy maximizando utilizacion pallet | Ordenamiento según #cajas asignables",
                                  grosor=grosor)

MemoryError: Unable to allocate 132. MiB for an array with shape (4, 4327527) and data type object

In [ ]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy maximizando utilizacion pallet | Ordenamiento según #cajas asignables
Número de tipos de cajas distintos: 75
Costo packaging: 27425415.30000001
Costo flete: 161919750
Costo total: 189345165.3
Utilización de pallet promedio: 0.9551848877347696
Utilización de caja promedio: 0.9456430159717513
Ahorro costo total: 9.50602%


In [ ]:
relocate(solucion_greedy, productos, asignaciones_por_producto, 
         titulo_solucion="Greedy + Relocate maximizando utilizacion pallet")

Baja de  189345165.3 a  189343218.96
Baja de  189343218.96 a  189341824.32000002
Baja de  189341824.32000002 a  189337367.82000002
Baja de  189337367.82000002 a  189336812.70000002
Baja de  189336812.70000002 a  189333795.72
Baja de  189333795.72 a  189333470.70000002
Baja de  189333470.70000002 a  189324005.82000002
Baja de  189324005.82000002 a  189323214.48000002
Baja de  189323214.48000002 a  189323211.36
Baja de  189323211.36 a  189323204.28
Baja de  189323204.28 a  189323202.18
Baja de  189323202.18 a  189320621.82000002
Baja de  189320621.82000002 a  189310229.58
Baja de  189310229.58 a  189308146.68
Baja de  189308146.68 a  189307816.02
Baja de  189307816.02 a  189307806.78
Baja de  189307806.78 a  189298566.42000002
Baja de  189298566.42000002 a  189295870.14000002
Baja de  189295870.14000002 a  189294115.38000003
Baja de  189294115.38000003 a  189288584.16000003
Baja de  189288584.16000003 a  189280705.14000002
Baja de  189280705.14000002 a  189278813.04000002
Baja de  189278

In [ ]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy + Relocate maximizando utilizacion pallet
Número de tipos de cajas distintos: 59
Costo packaging: 27105592.32
Costo flete: 161910450
Costo total: 189016042.32
Utilización de pallet promedio: 0.9473819331772754
Utilización de caja promedio: 0.9537003520282602
Ahorro costo total: 9.66332%
